In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD # Added for SVD
import warnings
warnings.filterwarnings('ignore')

# ====================
# Load Data
# ====================
train_df = pd.read_csv("train (1).csv")
test_df = pd.read_csv("test.csv")
item_df = pd.read_csv("item.csv")
user_df = pd.read_csv("user.csv")

# ====================
# Occupation Mapping
# ====================
with open("occupation.txt") as f:
    occupation_list = [line.strip() for line in f]

if pd.api.types.is_numeric_dtype(user_df['occupation']):
    user_df['occupation'] = user_df['occupation'].map(
        lambda x: occupation_list[int(x)] if pd.notnull(x) else 'unknown'
    )
else:
    user_df['occupation'] = user_df['occupation'].fillna('unknown')

# ====================
# Data Preparation
# ====================
item_df.columns = item_df.columns.str.strip()
item_df.rename(columns={'movie_id': 'item_id'}, inplace=True)

# Merge
train_df = train_df.merge(user_df[['user_id','age','gender','occupation']], on='user_id', how='left')
test_df = test_df.merge(user_df[['user_id','age','gender','occupation']], on='user_id', how='left')
train_df = train_df.merge(item_df, on='item_id', how='left')
test_df = test_df.merge(item_df, on='item_id', how='left')

# ====================
# SVD Feature Engineering (NEW SECTION)
# ====================
print("Creating SVD features...")
# Create a user-item interaction matrix
n_users = train_df.user_id.nunique()
n_items = train_df.item_id.nunique()

# Use csr_matrix for efficient storage
user_item_matrix = csr_matrix((train_df['rating'], (train_df['user_id'], train_df['item_id'])))

# Apply TruncatedSVD to create latent features (embeddings)
n_components = 25 # Number of latent features to create
svd = TruncatedSVD(n_components=n_components, random_state=42)
svd.fit(user_item_matrix)

# Create user and item embeddings
user_features = svd.transform(user_item_matrix)
item_features = svd.components_.T

# Create dataframes for the new features
user_features_df = pd.DataFrame(user_features, columns=[f'user_svd_{i}' for i in range(n_components)])
user_features_df['user_id'] = np.arange(user_item_matrix.shape[0])

item_features_df = pd.DataFrame(item_features, columns=[f'item_svd_{i}' for i in range(n_components)])
item_features_df['item_id'] = np.arange(user_item_matrix.shape[1])

# Merge the new SVD features back into the main dataframes
train_df = train_df.merge(user_features_df, on='user_id', how='left')
test_df = test_df.merge(user_features_df, on='user_id', how='left')

train_df = train_df.merge(item_features_df, on='item_id', how='left')
test_df = test_df.merge(item_features_df, on='item_id', how='left')
print(f"{n_components*2} SVD features added successfully!")


# ====================
# Advanced Feature Engineering
# ====================
def extract_year(x):
    try: return int(str(x).split("-")[-1])
    except: return np.nan
train_df['release_year'] = train_df['release_date'].apply(extract_year)
test_df['release_year'] = test_df['release_date'].apply(extract_year)
median_year = train_df['release_year'].median()
train_df['release_year'].fillna(median_year, inplace=True)
test_df['release_year'].fillna(median_year, inplace=True)
train_df['movie_age'] = 1998 - train_df['release_year']
test_df['movie_age'] = 1998 - test_df['release_year']

# User and movie stats
user_stats = train_df.groupby('user_id').agg({'rating': ['mean', 'std', 'count']}).reset_index()
user_stats.columns = ['user_id', 'user_avg_rating', 'user_std_rating', 'user_rating_count']
movie_stats = train_df.groupby('item_id').agg({'rating': ['mean', 'std', 'count']}).reset_index()
movie_stats.columns = ['item_id', 'movie_avg_rating', 'movie_std_rating', 'movie_rating_count']
train_df = train_df.merge(user_stats, on='user_id', how='left')
test_df = test_df.merge(user_stats, on='user_id', how='left')
train_df = train_df.merge(movie_stats, on='item_id', how='left')
test_df = test_df.merge(movie_stats, on='item_id', how='left')

# Label Encoding for categorical features
# We do this before filling NaNs to keep consistency
for col in ['gender', 'occupation']:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    test_df[col] = test_df[col].map(lambda x: le.transform([str(x)])[0] if str(x) in le.classes_ else -1)

# Fill any remaining missing values
train_df.fillna(-999, inplace=True)
test_df.fillna(-999, inplace=True)

# ====================
# Feature Selection
# ====================
drop_cols = ['rating', 'timestamp', 'title', 'release_date', 'imdb_url']
feature_cols = [c for c in train_df.columns if c not in drop_cols]

X = train_df[feature_cols]
y = train_df['rating']
X_test = test_df[feature_cols]

# ====================
# Model Training with K-Fold CV
# ====================
n_splits = 7
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_preds_xgb = np.zeros(len(X))
oof_preds_lgb = np.zeros(len(X))
oof_preds_rf = np.zeros(len(X))
test_preds_xgb = np.zeros(len(X_test))
test_preds_lgb = np.zeros(len(X_test))
test_preds_rf = np.zeros(len(X_test))

fold = 1
for train_idx, val_idx in skf.split(X, y.astype(int)):
    print(f"\n{'='*50}\nFold {fold}/{n_splits}\n{'='*50}")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # ========== XGBoost ==========
    xgb_model = xgb.XGBRegressor(
        objective='reg:squarederror', eval_metric='rmse', max_depth=7,
        learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
        min_child_weight=3, reg_alpha=0.5, reg_lambda=2, n_estimators=1000,
        gamma=0.1, random_state=42, early_stopping_rounds=50
    )
    xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    val_preds_xgb = xgb_model.predict(X_val)
    oof_preds_xgb[val_idx] = val_preds_xgb
    test_preds_xgb += xgb_model.predict(X_test) / n_splits
    print(f"XGBoost RMSE: {np.sqrt(mean_squared_error(y_val, val_preds_xgb)):.4f}")

    # ========== LightGBM ==========
    lgb_model = lgb.LGBMRegressor(
        objective='regression', metric='rmse', num_leaves=31,
        learning_rate=0.03, feature_fraction=0.8, bagging_fraction=0.8,
        bagging_freq=5, min_child_samples=20, reg_alpha=0.5, n_estimators=1000,
        reg_lambda=2, random_state=42,
    )
    lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
    val_preds_lgb = lgb_model.predict(X_val)
    oof_preds_lgb[val_idx] = val_preds_lgb
    test_preds_lgb += lgb_model.predict(X_test) / n_splits
    print(f"LightGBM RMSE: {np.sqrt(mean_squared_error(y_val, val_preds_lgb)):.4f}")

    # ========== Random Forest ==========
    rf_model = RandomForestRegressor(
        n_estimators=200, max_depth=12, min_samples_leaf=10,
        random_state=42, n_jobs=-1, max_features=0.7
    )
    rf_model.fit(X_train, y_train)
    val_preds_rf = rf_model.predict(X_val)
    oof_preds_rf[val_idx] = val_preds_rf
    test_preds_rf += rf_model.predict(X_test) / n_splits
    print(f"Random Forest RMSE: {np.sqrt(mean_squared_error(y_val, val_preds_rf)):.4f}")

    # ========== Ensemble ==========
    val_preds_ensemble = (val_preds_xgb + val_preds_lgb + val_preds_rf) / 3.0
    print(f"Ensemble RMSE: {np.sqrt(mean_squared_error(y_val, val_preds_ensemble)):.4f}")
    fold += 1

# ====================
# Overall CV Scores
# ====================
print(f"\n{'='*50}\nOVERALL CROSS-VALIDATION SCORES\n{'='*50}")
print(f"XGBoost CV RMSE: {np.sqrt(mean_squared_error(y, oof_preds_xgb)):.4f}")
print(f"LightGBM CV RMSE: {np.sqrt(mean_squared_error(y, oof_preds_lgb)):.4f}")
print(f"Random Forest CV RMSE: {np.sqrt(mean_squared_error(y, oof_preds_rf)):.4f}")
oof_preds_ensemble = (oof_preds_xgb + oof_preds_lgb + oof_preds_rf) / 3.0
print(f"Ensemble CV RMSE: {np.sqrt(mean_squared_error(y, oof_preds_ensemble)):.4f}")

# ====================
# Final Predictions
# ====================
test_preds_ensemble = (test_preds_xgb + test_preds_lgb + test_preds_rf) / 3.0
y_test_pred_clipped = np.clip(np.round(test_preds_ensemble), 1, 5)

submission = pd.DataFrame({'id': test_df['id'], 'rating': y_test_pred_clipped.astype(int)})
submission.to_csv('IITG_Roll_number_Name2.csv', index=False)
print(f"\n{'='*50}\nPredictions saved to 'IITG_Roll_number_Name2.csv'\n{'='*50}")

Creating SVD features...
50 SVD features added successfully!

Fold 1/7
XGBoost RMSE: 0.8477
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.018793 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14824
[LightGBM] [I